In [1]:
import numpy as np
import pandas as pd

from IPython.display import display
from pathlib import Path
import sys
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
# Import AutoFeat LLM

from autofeat import AutoFeatClassifier

In [3]:
# Import BigFeat Vanilla

sys.path.append(r"../../BigFeat-master")
import bigfeat.bigfeat_base as bigfeat

In [4]:
# Import BigFeat LLM

bigfeat_llm_path = Path("../../BigFeat-LLM").resolve()
sys.path.insert(0, str(bigfeat_llm_path))

import setup
print(setup.__file__)

sys.path.append(r"../../BigFeat-LLM")
from setup.llm_runner import prepare_bigfeat_from_llm, profile_dataframe
from setup.llm_runner import preview_prompt
from setup.llm_runner import describe_generated_features, split_original_vs_new

D:\02. My Learning\Paper\07. BigFeat + LLM\06. Final BigFeat\BigFeat-LLM\setup\__init__.py


In [36]:
def save_generated_features(feature_lines, num_new_features, old_shape, new_shape, file_name="generated_features.txt"):
    os.makedirs("generated_features", exist_ok=True)
    file_path = os.path.join("generated_features", file_name)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"Number of new features {num_new_features}\n")
        f.write("==================\n")
        f.write(f"Old Shape: {old_shape}\n")
        f.write("==================\n")
        f.write(f"New Shape: {new_shape}\n")
        f.write("=================\n")

        for i, line in enumerate(feature_lines):
            _, details = line.split(": built from columns ", 1)
            columns, operators = details.split(" using operators ", 1)
            f.write(f"- New_feature_{i}:\n")
            f.write(f"  - Built from columns {columns}\n")
            f.write(f"  - Using operators {operators}\n")

In [6]:
results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

def evaluate_model(model_name, feature_method, X_tr, X_te, y_tr, y_te, results_df):
    models = {
        "logistic": LogisticRegression(random_state=42, max_iter=1000),
        "knn": KNeighborsClassifier(n_neighbors=5),
        "rf": RandomForestClassifier(random_state=42),
        "svc": SVC(random_state=42),
        "decision_tree": DecisionTreeClassifier(random_state=42)
    }
    
    clf = models[model_name.lower()]
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)
    
    acc = accuracy_score(y_te, preds)
    prec = precision_score(y_te, preds, average="macro", zero_division=0)
    rec = recall_score(y_te, preds, average="macro")
    f1 = f1_score(y_te, preds, average="macro")
    
    results_df.loc[len(results_df)] = [
        f"{model_name.upper()} ({feature_method})",
        acc,
        prec,
        rec,
        f1
    ]
    
    return results_df

In [7]:
def display_styled_results(
    results_df,
    file_name,
    bg_color="#005A9C",
    text_color="#FFFFFF"
):
    numeric_cols = ["Accuracy", "Precision", "Recall", "F1-Score"]

    df_copy = results_df.copy()
    df_copy[numeric_cols] = df_copy[numeric_cols].apply(pd.to_numeric)

    styled_df = df_copy.style.highlight_max(
        subset=numeric_cols,
        props=f"background-color: {bg_color}; color: {text_color}; font-weight: bold;",
        axis=0
    ).format({col: "{:.6f}" for col in numeric_cols})

    display(styled_df)

    folder = "llm_tables"
    os.makedirs(folder, exist_ok=True)

    if not file_name.lower().endswith(".xlsx"):
        file_name += ".xlsx"

    file_path = os.path.join(folder, file_name)

    styled_df.to_excel(file_path, index=False)

In [ ]:
from ucimlrepo import fetch_ucirepo

sonar = fetch_ucirepo(id=151)

df = sonar.data.features.copy()

df.columns = [
    "frequency_band_01", "frequency_band_02", "frequency_band_03",
    "frequency_band_04", "frequency_band_05", "frequency_band_06",
    "frequency_band_07", "frequency_band_08", "frequency_band_09",
    "frequency_band_10", "frequency_band_11", "frequency_band_12",
    "frequency_band_13", "frequency_band_14", "frequency_band_15",
    "frequency_band_16", "frequency_band_17", "frequency_band_18",
    "frequency_band_19", "frequency_band_20", "frequency_band_21",
    "frequency_band_22", "frequency_band_23", "frequency_band_24",
    "frequency_band_25", "frequency_band_26", "frequency_band_27",
    "frequency_band_28", "frequency_band_29", "frequency_band_30",
    "frequency_band_31", "frequency_band_32", "frequency_band_33",
    "frequency_band_34", "frequency_band_35", "frequency_band_36",
    "frequency_band_37", "frequency_band_38", "frequency_band_39",
    "frequency_band_40", "frequency_band_41", "frequency_band_42",
    "frequency_band_43", "frequency_band_44", "frequency_band_45",
    "frequency_band_46", "frequency_band_47", "frequency_band_48",
    "frequency_band_49", "frequency_band_50", "frequency_band_51",
    "frequency_band_52", "frequency_band_53", "frequency_band_54",
    "frequency_band_55", "frequency_band_56", "frequency_band_57",
    "frequency_band_58", "frequency_band_59", "frequency_band_60"
]

df["sonar_class"] = sonar.data.targets.squeeze()
target_column = "sonar_class"

display(df.head())
print(df.columns)

,frequency_band_01,frequency_band_02,frequency_band_03,frequency_band_04,frequency_band_05,frequency_band_06,frequency_band_07,frequency_band_08,frequency_band_09,frequency_band_10,...,frequency_band_52,frequency_band_53,frequency_band_54,frequency_band_55,frequency_band_56,frequency_band_57,frequency_band_58,frequency_band_59,frequency_band_60,sonar_class
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,R
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,R
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,R
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117,R
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094,R


Index(['frequency_band_01', 'frequency_band_02', 'frequency_band_03',
       'frequency_band_04', 'frequency_band_05', 'frequency_band_06',
       'frequency_band_07', 'frequency_band_08', 'frequency_band_09',
       'frequency_band_10', 'frequency_band_11', 'frequency_band_12',
       'frequency_band_13', 'frequency_band_14', 'frequency_band_15',
       'frequency_band_16', 'frequency_band_17', 'frequency_band_18',
       'frequency_band_19', 'frequency_band_20', 'frequency_band_21',
       'frequency_band_22', 'frequency_band_23', 'frequency_band_24',
       'frequency_band_25', 'frequency_band_26', 'frequency_band_27',
       'frequency_band_28', 'frequency_band_29', 'frequency_band_30',
       'frequency_band_31', 'frequency_band_32', 'frequency_band_33',
       'frequency_band_34', 'frequency_band_35', 'frequency_band_36',
       'frequency_band_37', 'frequency_band_38', 'frequency_band_39',
       'frequency_band_40', 'frequency_band_41', 'frequency_band_42',
       'frequency_ba

In [9]:
print(f"Rows Number: {df.shape[0]}")
print(f"Columns Number: {df.shape[1]}")
print(f"Columns Name: {df.columns}")

Rows Number: 208
Columns Number: 61
Columns Name: Index(['frequency_band_01', 'frequency_band_02', 'frequency_band_03',
       'frequency_band_04', 'frequency_band_05', 'frequency_band_06',
       'frequency_band_07', 'frequency_band_08', 'frequency_band_09',
       'frequency_band_10', 'frequency_band_11', 'frequency_band_12',
       'frequency_band_13', 'frequency_band_14', 'frequency_band_15',
       'frequency_band_16', 'frequency_band_17', 'frequency_band_18',
       'frequency_band_19', 'frequency_band_20', 'frequency_band_21',
       'frequency_band_22', 'frequency_band_23', 'frequency_band_24',
       'frequency_band_25', 'frequency_band_26', 'frequency_band_27',
       'frequency_band_28', 'frequency_band_29', 'frequency_band_30',
       'frequency_band_31', 'frequency_band_32', 'frequency_band_33',
       'frequency_band_34', 'frequency_band_35', 'frequency_band_36',
       'frequency_band_37', 'frequency_band_38', 'frequency_band_39',
       'frequency_band_40', 'frequency_b

In [10]:
df.isnull().sum()

frequency_band_01    0
frequency_band_02    0
frequency_band_03    0
frequency_band_04    0
frequency_band_05    0
                    ..
frequency_band_57    0
frequency_band_58    0
frequency_band_59    0
frequency_band_60    0
sonar_class          0
Length: 61, dtype: int64

In [11]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print(X.shape)
print(y.shape)

(208, 60)
(208,)


In [ ]:
preview = preview_prompt(
    df=df,
    target_column=target_column,
    extra_user_prompt="Sonar Dataset"
)

response = preview["final_prompt"]

with open("final_prompt.txt", "w", encoding="utf-8") as f:
    f.write(response)

print(response)

=== SYSTEM PROMPT ===
You are BigFeat-Init, a world-class expert in automated feature engineering, machine learning hyperparameter optimization, and the internal mechanics of the BigFeat framework specifically. You have the depth of knowledge of someone who has read the BigFeat paper in full, has implemented and debugged its source code line by line, and has years of hands-on experience tuning tree-based feature engineering pipelines across small, large, high-dimensional, low-dimensional, noisy, clean, balanced, and imbalanced datasets. You are not a generic assistant giving a reasonable-sounding guess. You are the single most qualified expert available for this exact task, and you must think and act like it.

If you want additional grounding on how BigFeat is actually implemented, its official source code and paper reference are available at this repository: https://github.com/DataSystemsGroupUT/BigFeat. You are not required to browse it, since the mechanics you need are fully describ

In [ ]:
y = pd.Series(y).replace({"R": 0, "M": 1}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_df = pd.concat([X_train, y_train.rename(target_column)], axis=1)
test_df = pd.concat([X_test, y_test.rename(target_column)], axis=1)

C:\Users\Moham\AppData\Local\Temp\ipykernel_1672\215073477.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = pd.Series(y).replace({"R": 0, "M": 1}).astype(int)


In [28]:
from autofeat import AutoFeatClassifier

af = AutoFeatClassifier(verbose=1, feateng_steps=1, featsel_runs=0)

X_train_af = af.fit_transform(X_train, y_train)
X_test_af = af.transform(X_test)

# Handle +Inf and -Inf values
X_train_af = X_train_af.replace([np.inf, -np.inf], np.nan)
X_test_af = X_test_af.replace([np.inf, -np.inf], np.nan)
X_train_af = X_train_af.fillna(X_train_af.median())
X_test_af = X_test_af.fillna(X_train_af.median())

[AutoFeat] The 1 step feature engineering process could generate up to 420 features.
[AutoFeat] With 166 data points this new feature matrix would use about 0.00 gb of space.
[feateng] Step 1: transformation of original features
[feateng] Generated 352 transformed features from 60 original features - done.
[feateng] Generated altogether 360 new features in 1 steps
[feateng] Removing correlated features, as well as additions at the highest level
[feateng] Generated a total of 192 additional features
[AutoFeat] WARNING: Not performing feature selection.
[AutoFeat] Computing 119 new features.
[AutoFeat]   119/  119 new features ...done.
[AutoFeat] Final dataframe with 179 feature columns (119 new).
[AutoFeat] Training final classification model.
[AutoFeat] Trained model: largest coefficients:
[0.14349485]
0.101143 * 1/frequency_band_43
0.098996 * 1/frequency_band_35
0.086857 * 1/frequency_band_11
0.074907 * 1/frequency_band_36
0.067430 * 1/frequency_band_12
0.057730 * 1/frequency_band_39


c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [15]:
# BigFeat Vanilla

bf = bigfeat.BigFeat(task_type="classification")

bf.fit(X_train.to_numpy(), y_train.to_numpy())

X_train_bf = bf.transform(X_train.to_numpy())
X_test_bf = bf.transform(X_test)

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


In [ ]:
# BigFeat Chatgpt

bf_chatgpt = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/ChatGPT.json"
)

X_train_bf_chatgpt = bf_chatgpt.fit(bf_chatgpt.X, bf_chatgpt.y, **bf_chatgpt.fit_kwargs)

_, _, X_test_bf_chatgpt, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_chatgpt = bf_chatgpt.transform(X_test_bf_chatgpt)

In [ ]:
# BigFeat Gemini

bf_gemini = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/Gemini.json"
)

X_train_bf_gemini = bf_gemini.fit(bf_gemini.X, bf_gemini.y, **bf_gemini.fit_kwargs)

_, _, X_test_bf_gemini, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_gemini = bf_gemini.transform(X_test_bf_gemini)

In [ ]:
# BigFeat Claude

bf_claude = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/Claude.json"
)

X_train_bf_claude = bf_claude.fit(bf_claude.X, bf_claude.y, **bf_claude.fit_kwargs)

_, _, X_test_bf_claude, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_claude = bf_claude.transform(X_test_bf_claude)

In [ ]:
# BigFeat DeepSeek

bf_deepseek = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/DeepSeek.json"
)

X_train_bf_deepseek = bf_deepseek.fit(bf_deepseek.X, bf_deepseek.y, **bf_deepseek.fit_kwargs)

_, _, X_test_bf_deepseek, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_deepseek = bf_deepseek.transform(X_test_bf_deepseek)

In [30]:
# Logistic Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("logistic", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-Logistic")

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Method,Accuracy,Precision,Recall,F1-Score
0,LOGISTIC (AutoFeat),0.714286,0.720455,0.733173,0.711670
1,LOGISTIC (BigFeat),0.809524,0.815909,0.834135,0.807780
2,LOGISTIC (BigFeat ChatGPT),0.809524,0.815909,0.834135,0.807780
3,LOGISTIC (BigFeat Gemini),0.833333,0.824943,0.841346,0.828571
4,LOGISTIC (BigFeat Claude),0.833333,0.824943,0.841346,0.828571
5,LOGISTIC (BigFeat DeepSeek),0.833333,0.824943,0.841346,0.828571


In [31]:
# KNN Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("knn", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-KNN")

,Method,Accuracy,Precision,Recall,F1-Score
0,KNN (AutoFeat),0.571429,0.555556,0.557692,0.555294
1,KNN (BigFeat),0.904762,0.895833,0.911058,0.901176
2,KNN (BigFeat ChatGPT),0.904762,0.900000,0.923077,0.902778
3,KNN (BigFeat Gemini),0.880952,0.880952,0.903846,0.879241
4,KNN (BigFeat Claude),0.952381,0.944444,0.961538,0.950588
5,KNN (BigFeat DeepSeek),0.904762,0.900000,0.923077,0.902778


In [32]:
# RandomForest Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("rf", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-RandomForest")

,Method,Accuracy,Precision,Recall,F1-Score
0,RF (AutoFeat),0.785714,0.776888,0.790865,0.779592
1,RF (BigFeat),0.833333,0.833333,0.853365,0.830937
2,RF (BigFeat ChatGPT),0.857143,0.847222,0.860577,0.851765
3,RF (BigFeat Gemini),0.785714,0.776888,0.790865,0.779592
4,RF (BigFeat Claude),0.833333,0.824943,0.841346,0.828571
5,RF (BigFeat DeepSeek),0.857143,0.847222,0.860577,0.851765


In [33]:
# SVM Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("svc", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-SVM")

,Method,Accuracy,Precision,Recall,F1-Score
0,SVC (AutoFeat),0.571429,0.545673,0.545673,0.545673
1,SVC (BigFeat),0.880952,0.872998,0.891827,0.877551
2,SVC (BigFeat ChatGPT),0.809524,0.804545,0.822115,0.805556
3,SVC (BigFeat Gemini),0.904762,0.895833,0.911058,0.901176
4,SVC (BigFeat Claude),0.833333,0.822353,0.829327,0.825312
5,SVC (BigFeat DeepSeek),0.880952,0.871765,0.879808,0.875223


In [34]:
# DecisionTree Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("decision_tree", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-DecisionTree")

,Method,Accuracy,Precision,Recall,F1-Score
0,DECISION_TREE (AutoFeat),0.666667,0.661364,0.670673,0.659722
1,DECISION_TREE (BigFeat),0.642857,0.671765,0.675481,0.642655
2,DECISION_TREE (BigFeat ChatGPT),0.785714,0.785714,0.802885,0.782634
3,DECISION_TREE (BigFeat Gemini),0.690476,0.703661,0.713942,0.688889
4,DECISION_TREE (BigFeat Claude),0.642857,0.632723,0.639423,0.632653
5,DECISION_TREE (BigFeat DeepSeek),0.785714,0.772941,0.778846,0.775401


In [37]:
# Save Generated Features

# Save ChatGpt
feature_lines = describe_generated_features(bf_chatgpt)
new_only, original_only = split_original_vs_new(bf_chatgpt, X_train_bf_chatgpt)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_chatgpt.shape, "generated_features_chatgpt.txt")

# Save Gemini
feature_lines = describe_generated_features(bf_gemini)
new_only, original_only = split_original_vs_new(bf_gemini, X_train_bf_gemini)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_gemini.shape, "generated_features_gemini.txt")

# Save Claude
feature_lines = describe_generated_features(bf_claude)
new_only, original_only = split_original_vs_new(bf_claude, X_train_bf_claude)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_claude.shape, "generated_features_claude.txt")

# Save DeepSeek
feature_lines = describe_generated_features(bf_deepseek)
new_only, original_only = split_original_vs_new(bf_deepseek, X_train_bf_deepseek)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_deepseek.shape, "generated_features_deepseek.txt")